# 00 — HRI(t) to HRI(t+1): data preparation

Run the cells in order. All dataset construction code is visible here. The default reads the completed v2 dataset and audits without changing it. `RUN_PREPARATION` enables a new preparation run; a completed dataset version is protected.

The primary index retains four equal 0.25 weights. These are transparent analytical weights awaiting ecological expert review. Later-year and separate-block evaluation starts after the 2010–2014 label calibration period.

## 1 — Imports, paths, configuration, and input allowlist

In [1]:
import hashlib
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = globals().get('ROOT')
if ROOT is None:
    ROOT = next((p for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                 for p in [base, base / 'components/habitat_risk']
                 if (p / 'configs/hri.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Launch Jupyter inside the project repository.')
NOTEBOOK_DIR = ROOT / 'notebooks/hri/HRI_t_plus_1'
# A completed experiment is frozen. Use a new name for further development.
DATA_VERSION = 'hri_t_plus_1_v2'
EXPERIMENT_NAME = 'hri_t_plus_1_v2'

DATA = ROOT / "data/processed/hri" / DATA_VERSION
ARTIFACTS = ROOT / "artifacts/hri" / EXPERIMENT_NAME
KEY = ["cell_id", "reference_year"]
CORE = ["forest_amount_risk", "recent_loss_risk", "cumulative_loss_risk", "fragmentation_risk"]
CONFIG = json.loads((ROOT / "configs/hri.json").read_text())
WEIGHTS = CONFIG["weights"]
THRESHOLDS = CONFIG["risk_class_thresholds"]
RAW_FEATURES = [
    "forest__remaining_baseline_forest_proportion", "forest__annual_loss_fraction_of_baseline",
    "forest__cumulative_prior_loss_fraction_of_baseline", "forest__previous_3yr_loss_fraction_of_baseline",
    "forest__previous_5yr_loss_fraction_of_baseline", "forest__baseline_canopy_percent",
    "fragmentation__cell_patch_count", "fragmentation__mean_cell_patch_ha",
    "fragmentation__edge_m_per_land_ha", "terrain__mean_elevation_m",
    "terrain__mean_slope_degrees", "climate__annual_mean_temperature_c",
    "climate__annual_precipitation_mm",
]
FEATURES = RAW_FEATURES + CORE + [
    "hri_score_t", "hri_change_previous_year", "annual_loss_fraction_lag1",
    "annual_loss_fraction_lag2", "forest_proportion_change_previous_year",
    "edge_density_change_previous_year", "known_recent_loss_risk_t1",
    "known_cumulative_loss_risk_t1", "transition_hri_baseline",
]
TARGET = "target_hri_score_t1"


## 2 — Released class thresholds and source checksums

In [2]:
def risk_class(scores):
    return pd.cut(pd.Series(scores), [-np.inf, THRESHOLDS["low_upper"], THRESHOLDS["moderate_upper"], np.inf],
                  labels=["Low", "Moderate", "High"], right=True).astype("string")

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


## 3 — Add exact next-calendar-year labels

In [3]:
def add_next_year_targets(frame, scores):
    """Exact cell/year join; never shift across a missing calendar year."""
    if scores.duplicated(KEY).any() or frame.duplicated(KEY).any():
        raise ValueError("Duplicate cell-year keys")
    future = scores[KEY + ["hri_score_0_100"]].copy()
    future["reference_year"] -= 1
    future = future.rename(columns={"hri_score_0_100": TARGET})
    added = [TARGET, "target_year_t1", "target_risk_class_t1", "target_hri_change_t1", "target_hri_t1_available"]
    result = frame.drop(columns=added, errors="ignore").merge(future, on=KEY, how="left", validate="one_to_one")
    result["target_year_t1"] = result.reference_year + 1
    result["target_risk_class_t1"] = risk_class(result[TARGET]).to_numpy()
    result["target_hri_change_t1"] = result[TARGET] - result["hri_score_0_100"]
    result["target_hri_t1_available"] = result[TARGET].notna()
    return result


## 4 — Define 25 km blocks

In [4]:
def spatial_blocks(cell_ids, block_cells=5):
    # Grid IDs contain integer 5 km x/y coordinates. A block is 25 km x 25 km.
    xy = cell_ids.str.extract(r"_(-?\d+)_(-?\d+)$").astype(int)
    return (xy[0] // block_cells).astype(str) + ":" + (xy[1] // block_cells).astype(str)


## 5 — Construct past-only predictors and audit loss-window alignment

In [5]:
def build_frame(master, hri, scales):
    """Use released pre-2015 core calibration and exact past-only feature joins."""
    if master.duplicated(KEY).any() or hri.duplicated(KEY).any():
        raise ValueError("Source has duplicate cell-year keys")
    if set(WEIGHTS) != set(CORE) or min(WEIGHTS.values()) < 0 or not np.isclose(sum(WEIGHTS.values()), 1):
        raise ValueError("Invalid HRI weights")
    expected = hri[CORE].mul(pd.Series(WEIGHTS)).sum(axis=1) * 100
    if not np.allclose(expected, hri.hri_score_0_100, atol=1e-9):
        raise ValueError("Configured weights disagree with released HRI: version/recalculate before modelling")
    frame = hri[KEY + CORE + ["hri_score_0_100"]].merge(
        master[KEY + RAW_FEATURES], on=KEY, how="left", validate="one_to_one")
    frame = add_next_year_targets(frame, hri)
    # Lagged inputs are joined from all source years, not from a filtered split.
    for lag in [1, 2]:
        old = master[KEY + ["forest__annual_loss_fraction_of_baseline",
                            "forest__remaining_baseline_forest_proportion",
                            "fragmentation__edge_m_per_land_ha"]].copy()
        old["reference_year"] += lag
        old.columns = KEY + [f"annual_loss_fraction_lag{lag}", f"forest_proportion_lag{lag}", f"edge_density_lag{lag}"]
        frame = frame.merge(old, on=KEY, how="left", validate="one_to_one")
    previous_hri = hri[KEY + ["hri_score_0_100"]].copy()
    previous_hri.reference_year += 1
    previous_hri = previous_hri.rename(columns={"hri_score_0_100": "hri_score_previous_year"})
    frame = frame.merge(previous_hri, on=KEY, how="left", validate="one_to_one")
    frame["hri_score_t"] = frame.hri_score_0_100
    frame["hri_change_previous_year"] = frame.hri_score_t - frame.hri_score_previous_year
    frame["forest_proportion_change_previous_year"] = frame[RAW_FEATURES[0]] - frame.forest_proportion_lag1
    frame["edge_density_change_previous_year"] = frame["fragmentation__edge_m_per_land_ha"] - frame.edge_density_lag1
    # HRI at t+1 uses prior loss through t; these two components are already known at end of t.
    recent = frame[["forest__annual_loss_fraction_of_baseline", "annual_loss_fraction_lag1",
                    "annual_loss_fraction_lag2"]].sum(axis=1, min_count=3)
    cumulative = frame["forest__cumulative_prior_loss_fraction_of_baseline"] + frame["forest__annual_loss_fraction_of_baseline"]
    frame["known_recent_loss_risk_t1"] = (np.log1p(recent.clip(lower=0)) / np.log1p(scales["recent_loss_p99_2010_2014"])).clip(0, 1)
    frame["known_cumulative_loss_risk_t1"] = (np.log1p(cumulative.clip(lower=0)) / np.log1p(scales["cumulative_loss_p99_2010_2014"])).clip(0, 1)
    frame["transition_hri_baseline"] = 100 * (
        frame.forest_amount_risk * WEIGHTS["forest_amount_risk"] +
        frame.known_recent_loss_risk_t1 * WEIGHTS["recent_loss_risk"] +
        frame.known_cumulative_loss_risk_t1 * WEIGHTS["cumulative_loss_risk"] +
        frame.fragmentation_risk * WEIGHTS["fragmentation_risk"])
    # Cross-check the predictable components against released next-year components; never use them as X.
    future_components = hri[KEY + ["recent_loss_risk", "cumulative_loss_risk"]].copy()
    future_components.reference_year -= 1
    future_components.columns = KEY + ["audit_recent_t1", "audit_cumulative_t1"]
    audit = frame.merge(future_components, on=KEY, how="left", validate="one_to_one")
    for actual, known in [("audit_recent_t1", "known_recent_loss_risk_t1"),
                          ("audit_cumulative_t1", "known_cumulative_loss_risk_t1")]:
        ok = audit[actual].notna() & audit[known].notna()
        if not np.allclose(audit.loc[ok, actual], audit.loc[ok, known], atol=1e-8):
            raise ValueError(f"As-of-t loss-window alignment failed: {known}")
    frame = frame.loc[frame.target_year_t1.ge(2015) & frame[TARGET].notna()].copy()
    frame["spatial_block"] = spatial_blocks(frame.cell_id)
    blocks = np.array(sorted(frame.spatial_block.unique()))
    np.random.default_rng(42).shuffle(blocks)
    n_train, n_val = round(len(blocks) * .7), round(len(blocks) * .15)
    assignments = {b: "train" if i < n_train else "validation" if i < n_train+n_val else "test"
                   for i, b in enumerate(blocks)}
    frame["spatial_partition"] = frame.spatial_block.map(assignments)
    frame["time_partition"] = np.select([frame.target_year_t1.le(2019), frame.target_year_t1.le(2021)],
                                        ["train", "validation"], default="test")
    frame[FEATURES] = frame[FEATURES].replace([np.inf, -np.inf], np.nan)
    if frame[["hri_score_t", "transition_hri_baseline", TARGET]].isna().any().any():
        raise ValueError("Missing target or baseline")
    return frame.sort_values(["reference_year", "cell_id"]).reset_index(drop=True)


## 6 — Prepare outputs, weights sensitivity, backups, and target annotations

In [6]:
def build(update_existing=True):
    if (ROOT / "artifacts/hri" / DATA_VERSION / "final_test_manifest.json").exists():
        raise FileExistsError("The source dataset version is frozen. Use a new DATA_VERSION.")
    if (ARTIFACTS / "final_test_manifest.json").exists():
        raise FileExistsError("The v2 experiment is tested and frozen; use a new experiment version")
    sources = {"master": ROOT / "data/processed/hri/pre_hri_v1/master_cell_year_dataset.csv",
               "hri": ROOT / "data/processed/hri/hri_v1/hri_core_equal_complete_cases.csv",
               "audit": ROOT / "data/processed/hri/hri_v1/hri_audit.json"}
    master = pd.read_csv(sources["master"], low_memory=False)
    hri = pd.read_csv(sources["hri"])
    scales = json.loads(sources["audit"].read_text())["scale_parameters"]
    frame = build_frame(master, hri, scales)
    DATA.mkdir(parents=True, exist_ok=True)
    frame.to_csv(DATA / "forecast_dataset.csv", index=False)
    manifest = {"rows": len(frame), "cells": int(frame.cell_id.nunique()), "features": FEATURES,
                "weights": WEIGHTS, "weights_status": CONFIG["weights_status"], "thresholds": THRESHOLDS,
                "input_sha256": {k: sha256(v) for k, v in sources.items()},
                "dataset_sha256": sha256(DATA / "forecast_dataset.csv"),
                "split_counts": frame.groupby(["spatial_partition", "time_partition"]).size().to_dict(),
                "scope": "Retrospective constructed HRI forecasting; historical sources and previously explored data reused",
                "forecast_origin": "end of feature year t; annual t measurements must be available",
                "spatial_caveat": "25km blocks without buffers; dependence across neighboring block boundaries remains possible",
                "target_calibration": "Frozen national 2010-2014 label scales; distinct from training-fitted preprocessing"}
    manifest["split_counts"] = {"/".join(k): int(v) for k, v in manifest["split_counts"].items()}
    (DATA / "dataset_audit.json").write_text(json.dumps(manifest, indent=2))
    frame[FEATURES].isna().mean().rename("missing_fraction").to_csv(DATA / "feature_missingness.csv")
    # Describe alternatives on the training cohort only; never tune the scientific label to scores.
    hist = hri.loc[hri.reference_year.between(2015, 2019)].copy()
    hist = hist.merge(frame[["cell_id", "spatial_partition"]].drop_duplicates(), on="cell_id")
    hist = hist.loc[hist.spatial_partition.eq("train")]
    sensitivity = []
    for name, weights in {"core_equal": [.25]*4, "condition_emphasis": [.4,.2,.2,.2],
                          "disturbance_emphasis": [.15,.35,.30,.20]}.items():
        alternative = hist[CORE].mul(weights).sum(axis=1)*100
        sensitivity.append({"scenario": name, "weights": str(weights),
                            "rank_correlation_with_primary": hist.hri_score_0_100.corr(alternative, method="spearman"),
                            "mean_absolute_score_change": float((alternative-hist.hri_score_0_100).abs().mean()),
                            "class_changed_fraction": float((risk_class(alternative).to_numpy()!=risk_class(hist.hri_score_0_100).to_numpy()).mean())})
    pd.DataFrame(sensitivity).to_csv(DATA / "weight_sensitivity_training_only.csv", index=False)
    if update_existing:
        paths = [ROOT / "data/processed/hri/model_v1/final_hri_model_dataset.csv"]
        paths += [ROOT / f"data/splits/hri/model_v1/{n}.csv" for n in ["train", "validation", "test"]]
        backup = DATA / "original_dataset_backups"
        backup.mkdir(exist_ok=True)
        for path in paths:
            old = pd.read_csv(path)
            before = sha256(path)
            target = backup / ("final_hri_model_dataset.csv" if path.name.startswith("final") else path.name)
            if not target.exists():
                target.write_bytes(path.read_bytes())
            updated = add_next_year_targets(old, hri)
            if len(old) != len(updated):
                raise ValueError("Annotation changed row count")
            updated.to_csv(path, index=False)
            print("Annotated", path, "prior sha256", before, flush=True)
    print(frame.groupby(["spatial_partition", "time_partition"]).size().to_string(), flush=True)
    return frame


## 7 — Prepare only when explicitly enabled

In [7]:
RUN_PREPARATION = False
if RUN_PREPARATION:
    prepared = build(update_existing=True)
else:
    print('Review mode: loading existing dataset audit.')


Review mode: loading existing dataset audit.


## 8 — Inspect the saved dataset and checks

In [8]:
from IPython.display import display
audit = json.loads((DATA / 'dataset_audit.json').read_text())
print('Rows:', audit['rows'], 'Cells:', audit['cells'])
print('Dataset checksum matches:', sha256(DATA / 'forecast_dataset.csv') == audit['dataset_sha256'])
display(pd.read_csv(DATA / 'forecast_dataset.csv', nrows=5))
display(pd.Series(audit['split_counts'], name='rows').to_frame())
display(pd.Series(WEIGHTS, name='weight').to_frame())
display(pd.read_csv(DATA / 'feature_missingness.csv'))
display(pd.read_csv(DATA / 'weight_sensitivity_training_only.csv'))


Rows: 22725 Cells: 2525
Dataset checksum matches: True


,cell_id,reference_year,forest_amount_risk,recent_loss_risk,cumulative_loss_risk,fragmentation_risk,hri_score_0_100,forest__remaining_baseline_forest_proportion,forest__annual_loss_fraction_of_baseline,forest__cumulative_prior_loss_fraction_of_baseline,forest__previous_3yr_loss_fraction_of_baseline,forest__previous_5yr_loss_fraction_of_baseline,forest__baseline_canopy_percent,fragmentation__cell_patch_count,fragmentation__mean_cell_patch_ha,fragmentation__edge_m_per_land_ha,terrain__mean_elevation_m,terrain__mean_slope_degrees,climate__annual_mean_temperature_c,climate__annual_precipitation_mm,target_hri_score_t1,target_year_t1,target_risk_class_t1,target_hri_change_t1,target_hri_t1_available,annual_loss_fraction_lag1,forest_proportion_lag1,edge_density_lag1,annual_loss_fraction_lag2,forest_proportion_lag2,edge_density_lag2,hri_score_previous_year,hri_score_t,hri_change_previous_year,forest_proportion_change_previous_year,edge_density_change_previous_year,known_recent_loss_risk_t1,known_cumulative_loss_risk_t1,transition_hri_baseline,spatial_block,spatial_partition,time_partition
0,lka_laea_5000m_-10_-1,2014,0.561627,0.443799,0.359489,0.989379,58.857338,0.438373,0.017414,0.081522,0.050629,0.057401,34.465526,454.0,2.347731,130.717500,101.995262,1.325720,26.067301,1879.975444,63.926313,2015,Moderate,5.068975,True,0.011803,0.446845,127.317905,0.036376,0.452587,125.401770,54.939804,58.857338,3.917534,-0.008472,3.399595,0.570870,0.432758,63.865854,-2:-1,train,train
1,lka_laea_5000m_-10_-10,2014,0.346129,0.019363,0.036697,0.942704,33.622343,0.653871,0.010006,0.008032,0.002157,0.004728,51.085035,89.0,18.415618,103.627996,100.367288,2.320102,25.568572,2467.604619,36.818554,2015,Moderate,3.196211,True,0.000688,0.660534,100.878609,0.000597,0.660992,100.472177,33.191019,33.622343,0.431324,-0.006663,2.749387,0.100890,0.082005,36.793199,-2:-2,train,train
2,lka_laea_5000m_-10_-11,2014,0.307848,0.010610,0.012228,0.917051,31.193436,0.692152,0.003763,0.002669,0.001181,0.002057,55.830321,85.0,20.260588,91.191112,113.823832,5.403485,25.448990,2618.907048,32.315175,2015,Low,1.121739,True,0.000263,0.694773,90.337397,0.000219,0.694956,90.265253,30.991796,31.193436,0.201640,-0.002622,0.853714,0.038060,0.029412,32.309281,-2:-3,test,train
3,lka_laea_5000m_-10_-12,2014,0.203716,0.037413,0.053823,0.844306,28.481431,0.796284,0.004398,0.011803,0.004172,0.005939,68.910703,47.0,42.575745,63.392996,141.557145,9.545307,25.448457,2619.034038,29.751086,2015,Low,1.269655,True,0.000451,0.799844,61.946268,0.002781,0.800209,61.898442,28.184103,28.481431,0.297328,-0.003560,1.446728,0.068304,0.073718,29.751086,-2:-3,test,train
4,lka_laea_5000m_-10_-13,2014,0.179815,0.025020,0.056698,0.840920,27.561328,0.820185,0.003082,0.012437,0.002788,0.005136,70.957526,50.0,41.198400,62.326439,133.901593,9.555316,25.307074,2725.895356,28.600721,2015,Low,1.039392,True,0.000660,0.822752,61.165374,0.001761,0.823302,60.686584,27.418921,27.561328,0.142407,-0.002567,1.161065,0.049314,0.070639,28.517223,-2:-3,test,train


,rows
test/test,800
test/train,2000
test/validation,800
train/test,3630
train/train,9075
train/validation,3630
validation/test,620
validation/train,1550
validation/validation,620


,weight
forest_amount_risk,0.25
recent_loss_risk,0.25
cumulative_loss_risk,0.25
fragmentation_risk,0.25


,Unnamed: 0,missing_fraction
0,forest__remaining_baseline_forest_proportion,0.000000
1,forest__annual_loss_fraction_of_baseline,0.000000
2,forest__cumulative_prior_loss_fraction_of_base...,0.000000
3,forest__previous_3yr_loss_fraction_of_baseline,0.000000
4,forest__previous_5yr_loss_fraction_of_baseline,0.000000
5,forest__baseline_canopy_percent,0.000000
6,fragmentation__cell_patch_count,0.000000
7,fragmentation__mean_cell_patch_ha,0.000000
8,fragmentation__edge_m_per_land_ha,0.000000
9,terrain__mean_elevation_m,0.025743


,scenario,weights,rank_correlation_with_primary,mean_absolute_score_change,class_changed_fraction
0,core_equal,"[0.25, 0.25, 0.25, 0.25]",1.000000,2.833754e-15,0.000000
1,condition_emphasis,"[0.4, 0.2, 0.2, 0.2]",0.984457,2.808513e+00,0.063581
2,disturbance_emphasis,"[0.15, 0.35, 0.3, 0.2]",0.979661,6.234694e+00,0.266226
